In [29]:
%display latex

In [30]:
# Ініціалізація змінних
var('t x h V gamma m t')

# --- 1. ЗАГАЛЬНІ ФІЗИЧНІ КОНСТАНТИ ТА ПАРАМЕТРИ РАКЕТИ (Falcon 1) ---
g0 = 9.80665           # Прискорення вільного падіння (м/с^2)
Re = 6371000.0         # Радіус Землі (м)
rho0 = 1.225           # Щільність повітря на рівні моря (кг/м^3)
H_scale = 8500.0       # Шкала висоти атмосфери (м)

m0 = 27670.0           # Стартова маса (кг)
T = 454000.0           # Тяга двигуна (Н)
Isp = 275.0            # Питомий імпульс (с)
A = 2.27               # Площа поперечного перерізу корпусу (м^2)
Cd = 0.3               # Коефіцієнт лобового опору
mu = T / (Isp * g0)    # Секундна витрата палива (кг/с)

In [31]:
# Загальні функції середовища та ракети
rho(h) = rho0 * exp(-h / H_scale)
g(h) = g0 * (Re / (Re + h))^2
D_body(h, V) = 0.5 * Cd * rho(h) * V^2 * A

In [32]:
# СИСТЕМА 1: КЛАСИЧНИЙ СТАРТ (Gravity Turn)# ==============================================================================

# Система диференціальних рівнянь координат, швидкості, кута та маси
eq_x1 = V * cos(gamma)
eq_h1 = V * sin(gamma)
eq_V1 = (T - D_body(h, V)) / m - g(h) * sin(gamma)
eq_gamma1 = -(g(h) / V) * cos(gamma) + (V / (Re + h)) * cos(gamma)
eq_m1 = -mu + 0*t 

# Початкові умови та розрахунок параметрів до повороту (методом Рунге-Кути 4-ого порядку)
ics_phase1 = [0, 0, 0.1, 10.0, 1.57, m0]
sol_phase1 = desolve_system_rk4(
    [eq_x1, eq_h1, eq_V1, eq_gamma1, eq_m1],
    [x, h, V, gamma, m],
    ics=ics_phase1, ivar=t, end_points=15, step=0.5
)

# Розрахунок для другої фази польоту після повороту (за початкові умови беруться результати попереднього розрахунку)
last_point = sol_phase1[-1]
t_end, x_end, h_end, V_end, gamma_end, m_end = last_point

# Імітуємо нахил вектора тяги системою управління до ~84.8 градусів
gamma_kick = 1.48 

# Задаємо початкові умови для другої фази польоту
ics_phase2 = [SR(t_end), SR(x_end), SR(h_end), SR(V_end), SR(gamma_kick), SR(m_end)]

# Розрахунок другої фази
sol_phase2 = desolve_system_rk4(
    [eq_x1, eq_h1, eq_V1, eq_gamma1, eq_m1],
    [x, h, V, gamma, m],
    ics=ics_phase2, ivar=t, end_points=170, step=0.5
)

# Об'єднуємо результати (відкидаємо останню точку першої фази, щоб не було дублікату на t=15)
sol_classic = sol_phase1[:-1] + sol_phase2

In [33]:
# СИСТЕМА 2: ВЕРОДИНАМІЧНИЙ СТАРТ, ПОЛІТ ПІД КУТОМ 45 (З ВИКОРИСТАННЯМ ПІДЙОМНОЇ СИЛИ)# ==============================================================================

K_wing = 4 # Аеродинамічная якість доданого крила
S_wing = 10 # Площа крила
Cl = 0.2 # Коефіцієнт підйомної сили крила

L_real(h, V) = 0.5 * Cl * rho(h) * V^2 * S_wing # Функція зміни підйомної сили у польоті
D_lift_real(h, V) = L_real(h, V) / K_wing # Функція індуктивного опору крила (опору, що виникає при обтіканні крила, в наслідок вихорів на закінцівках, де змішуються потоки повітря, обтікаючі крило)
D_total(h, V) = D_body(h, V) + D_lift_real(h, V) # Сумарна дія сили опору

# Система диференціальних рівнянь координат, швидкості, кута та маси
eq_x2 = V * cos(gamma)
eq_h2 = V * sin(gamma)
eq_V2 = (T - D_total(h, V)) / m - g(h) * sin(gamma)
eq_gamma2 = L_real(h, V) / (m * V) - (g(h) / V) * cos(gamma) + (V / (Re + h)) * cos(gamma) 
eq_m_winged = -mu + 0*t

# Розрахунок
ics_45_phase1 = [0, 0, 0.1, 10.0, 1.57, m0]

sol_45_phase1 = desolve_system_rk4(
    [eq_x2, eq_h2, eq_V2, eq_gamma2, eq_m2],
    [x, h, V, gamma, m],
    ics=ics_45_phase1, ivar=t, end_points=25, step=0.5
)
# Задаємо початкові умови для другої фази польоту, задаємо кут розвороту
last_point_45 = sol_45_phase1[-1]
t_end_45, x_end_45, h_end_45, V_end_45, gamma_end_45, m_end_45 = last_point_45
gamma_kick = 0.785
ics_45_phase2 = [SR(t_end_45), SR(x_end_45), SR(h_end_45), SR(V_end_45), SR(gamma_kick), SR(m_end_45)]

# Розрахунок другої фази
sol_45_phase2 = desolve_system_rk4(
    [eq_x2, eq_h2, eq_V2, eq_gamma2, eq_m2], 
    [x, h, V, gamma, m],
    ics=ics_45_phase2, ivar=t, end_points=120, step=0.5
)

# Об'єднуємо результати
sol_45 = sol_phase1 + sol_phase2[1:]